In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================
# ✅ Install dependencies
# ============================
!pip install flask pyngrok --quiet


In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [ ]:
import os
import shutil

drive_product_path = "/content/drive/MyDrive/vit_model/plant_products/products"
local_product_path = "static/products"

os.makedirs(local_product_path, exist_ok=True)

for file in os.listdir(drive_product_path):
    shutil.copy(
        os.path.join(drive_product_path, file),
        os.path.join(local_product_path, file)
    )

print("All product images copied to static/products")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/vit_model/plant_products/products'

In [ ]:
%%writefile app.py
import os
import json
import torch
import timm
import re
import base64
import numpy as np

from flask import Flask, request, render_template, send_from_directory
from werkzeug.utils import secure_filename
from PIL import Image
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

# ─────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────
UPLOAD_FOLDER      = "uploads"
ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg"}

VIT_MODEL_PATH = "/content/drive/MyDrive/vit_model/plant_vit_base.pth"
CLASS_PATH     = "/content/drive/MyDrive/vit_model/class_names.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# Thresholds — tune these if needed
CONFIDENCE_THRESHOLD = 40.0   # top-1 must be ≥ 40% confident
ENTROPY_THRESHOLD    = 3.2    # Shannon entropy must be below this
                               # (38-class uniform = 5.25, real leaf ~0.5–2.5)

def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS

def load_class_names(path):
    with open(path, "r") as f:
        return json.load(f)

# ─────────────────────────────────────────────────────────────────────────
#  TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────
inference_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ─────────────────────────────────────────────────────────────────────────
#  LOAD ViT-Base disease model
# ─────────────────────────────────────────────────────────────────────────
class_names = load_class_names(CLASS_PATH)

def build_vit_model(num_classes):
    return timm.create_model(
        "vit_base_patch16_224",
        pretrained=False,
        num_classes=num_classes
    )

vit_model = build_vit_model(len(class_names))
state     = torch.load(VIT_MODEL_PATH, map_location=DEVICE)
vit_model.load_state_dict(state)
vit_model.to(DEVICE)
vit_model.eval()

# ─────────────────────────────────────────────────────────────────────────
#  COLOUR ANALYSIS — fast check for green / brown plant tones
#  Runs first, lightweight (no model needed)
# ─────────────────────────────────────────────────────────────────────────
def has_plant_colours(pil_img):
    """
    Returns True if image contains ≥12% green/yellow-green/brown pixels.
    Resizes to 64×64 for speed (~0.001s).
    """
    arr = np.array(pil_img.resize((64, 64))).astype(float)
    if arr.ndim < 3:
        return False
    R, G, B = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

    # Green / healthy leaf
    green  = (G > R + 8) & (G > B + 8) & (G > 40)
    # Yellow-green / diseased leaf
    yellow = (R > 130) & (G > 130) & (B < 90) & (G >= R - 35)
    # Brown / dry or dead leaf
    brown  = (R > G + 12) & (R > B + 12) & (R > 70) & (R < 210)

    plant_ratio = (green | yellow | brown).sum() / (64 * 64)
    return float(plant_ratio) >= 0.12

# ─────────────────────────────────────────────────────────────────────────
#  ViT CONFIDENCE + ENTROPY VALIDATOR
#
#  The ViT-Base model was trained ONLY on plant leaf images.
#  When given a non-leaf image it produces a flat/uncertain probability
#  distribution because it has never seen such patterns.
#
#  We measure:
#    • Top-1 confidence  — low if image is unfamiliar
#    • Shannon entropy   — high if distribution is flat (uncertain)
#
#  A real leaf image:   confidence ≥ 70%,  entropy ≈ 0.3 – 2.5
#  A non-leaf image:    confidence < 40%,  entropy > 3.2
# ─────────────────────────────────────────────────────────────────────────
def vit_is_confident(probs_tensor):
    """
    probs_tensor: torch.Tensor shape [num_classes], already softmaxed.
    Returns (is_confident: bool, top1_conf: float, entropy: float)
    """
    probs_np   = probs_tensor.cpu().numpy()
    top1_conf  = float(probs_np.max()) * 100          # percent
    # Shannon entropy: H = -sum(p * log(p))
    eps        = 1e-9
    entropy    = float(-np.sum(probs_np * np.log(probs_np + eps)))

    is_confident = (top1_conf >= CONFIDENCE_THRESHOLD) and (entropy <= ENTROPY_THRESHOLD)
    return is_confident, top1_conf, entropy

# ─────────────────────────────────────────────────────────────────────────
#  PRODUCT CATALOGUE + ALIAS DICTIONARY
# ─────────────────────────────────────────────────────────────────────────
product_images = {
    "Ridomil Gold":       "ridomil_gold.jpg",
    "Saaf Fungicide":     "saaf.jpg",
    "Antracol Fungicide": "antracol.jpg",
    "Nativo Fungicide":   "nativo.jpg",
    "Mancozeb":           "mancozeb.jpg",
    "Copper Fungicide":   "copper_fungicide.jpg",
    "Neem Oil":           "neem_oil.jpg",
    "Confidor":           "confidor.jpg",
    "Actara":             "actara.jpg",
}

PRODUCT_ALIASES = {
    "copper":          "Copper Fungicide",
    "bordeaux":        "Copper Fungicide",
    "cupric":          "Copper Fungicide",
    "mancozeb":        "Mancozeb",
    "dithane":         "Mancozeb",
    "indofil":         "Mancozeb",
    "zineb":           "Mancozeb",
    "ridomil":         "Ridomil Gold",
    "metalaxyl":       "Ridomil Gold",
    "apron":           "Ridomil Gold",
    "saaf":            "Saaf Fungicide",
    "carbendazim":     "Saaf Fungicide",
    "bavistin":        "Saaf Fungicide",
    "captan":          "Saaf Fungicide",
    "thiram":          "Saaf Fungicide",
    "antracol":        "Antracol Fungicide",
    "propineb":        "Antracol Fungicide",
    "chlorothalonil":  "Antracol Fungicide",
    "sulphur":         "Antracol Fungicide",
    "sulfur":          "Antracol Fungicide",
    "nativo":          "Nativo Fungicide",
    "tebuconazole":    "Nativo Fungicide",
    "trifloxystrobin": "Nativo Fungicide",
    "propiconazole":   "Nativo Fungicide",
    "bayleton":        "Nativo Fungicide",
    "azoxystrobin":    "Nativo Fungicide",
    "hexaconazole":    "Nativo Fungicide",
    "neem":            "Neem Oil",
    "azadirachtin":    "Neem Oil",
    "confidor":        "Confidor",
    "imidacloprid":    "Confidor",
    "admire":          "Confidor",
    "spinosad":        "Confidor",
    "chlorpyrifos":    "Confidor",
    "dimethoate":      "Confidor",
    "actara":          "Actara",
    "thiamethoxam":    "Actara",
    "cruiser":         "Actara",
    "abamectin":       "Actara",
    "cypermethrin":    "Actara",
    "malathion":       "Actara",
}

def extract_products(text):
    found      = []
    seen       = set()
    text_lower = text.lower()
    for keyword, product in PRODUCT_ALIASES.items():
        if re.search(r'\b' + re.escape(keyword) + r'\b', text_lower):
            if product not in seen:
                seen.add(product)
                found.append(product)
    for product in product_images:
        if product not in seen and re.search(re.escape(product), text, re.IGNORECASE):
            seen.add(product)
            found.append(product)
    return found

# ─────────────────────────────────────────────────────────────────────────
#  LOAD MISTRAL-7B LLM
# ─────────────────────────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

LLM_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer    = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model    = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
llm_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=400,
    temperature=0.5,
    do_sample=True
)

def generate_prevention_text(disease_name):
    prompt = f"""You are an agricultural expert advising a farmer.

Plant disease detected: {disease_name}

Provide a structured response with exactly these sections:

• Disease Description:
• Causes:
• Symptoms:
• Prevention:
• Organic Treatment:
  - Include: Neem Oil, Copper Fungicide, Bordeaux mixture, Sulphur if applicable
• Chemical Treatment:
  - Include specific product brand names such as: Mancozeb, Ridomil Gold, Saaf Fungicide, Antracol Fungicide, Nativo Fungicide, Confidor, Actara as applicable
  - Mention active ingredients alongside brand names

Keep each section concise. Do NOT repeat the disease name as a heading. Start directly with the answer."""

    result   = llm_pipeline(prompt)[0]["generated_text"]
    result   = result.replace(prompt, "").strip()
    products = extract_products(result)
    return result, products

# ─────────────────────────────────────────────────────────────────────────
#  FLASK APP
# ─────────────────────────────────────────────────────────────────────────
app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024
app.config["UPLOAD_FOLDER"]      = UPLOAD_FOLDER

@app.route("/", methods=["GET", "POST"])
def index():
    pred_label = None
    confidence = None
    filename   = None
    advice     = None
    products   = []
    error      = None
    not_a_crop = False

    if request.method == "POST":
        file       = request.files.get("file")
        image_data = request.form.get("image_data")

        # ── Camera capture ───────────────────────────────────────────────
        if image_data:
            if ',' not in image_data:
                error = "Invalid camera data. Please try again."
                return render_template("index.html", error=error)

            header, encoded = image_data.split(",", 1)

            if not encoded.strip():
                error = "Empty camera capture. Please try again."
                return render_template("index.html", error=error)

            ext = "jpg" if "jpeg" in header else "png"
            try:
                img_bytes = base64.b64decode(encoded)
            except Exception:
                error = "Failed to decode image. Please try again."
                return render_template("index.html", error=error)

            filename = f"camera_capture.{ext}"
            filepath = os.path.join(app.config["UPLOAD_FOLDER"], filename)
            with open(filepath, "wb") as f:
                f.write(img_bytes)

        # ── File upload ──────────────────────────────────────────────────
        elif file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            filepath = os.path.join(app.config["UPLOAD_FOLDER"], filename)
            file.save(filepath)

        else:
            return render_template("index.html")

        # ── Open image ───────────────────────────────────────────────────
        try:
            pil_img = Image.open(filepath).convert("RGB")
        except Exception:
            error = "Could not read the image. Please upload a valid file."
            return render_template("index.html", error=error, filename=filename)

        # ── LAYER 1: Fast colour check ───────────────────────────────────
        # If clearly green/brown → skip heavy ViT validation, go straight
        # to disease prediction. Saves ~0.3s for valid leaf images.
        colour_pass = has_plant_colours(pil_img)

        # ── LAYER 2: ViT inference (always runs) ─────────────────────────
        x = inference_tfms(pil_img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            outputs  = vit_model(x)
            probs    = torch.softmax(outputs, dim=1)[0]
            pred_idx = torch.argmax(probs).item()

        is_confident, top1_conf, entropy = vit_is_confident(probs)

        # ── DECISION: crop or not ────────────────────────────────────────
        #
        #  Accept if:
        #    • Colour check passes  AND  ViT confidence ≥ threshold
        #    • OR ViT is very confident regardless of colour (conf ≥ 60%)
        #
        #  Reject if:
        #    • ViT is uncertain (entropy too high OR confidence too low)
        #    • AND colour check also failed

        very_confident = top1_conf >= 60.0
        accept = (colour_pass and is_confident) or very_confident

        if not accept:
            return render_template(
                "index.html",
                not_a_crop=True,
                filename=filename,
                top1_conf=round(top1_conf, 1),
                entropy=round(entropy, 2)
            )

        # ── ViT-Base disease label ────────────────────────────────────────
        pred_label = class_names[pred_idx]
        confidence = top1_conf

        # ── LLM Advice ───────────────────────────────────────────────────
        advice, products = generate_prevention_text(pred_label)

    return render_template(
        "index.html",
        pred_label=pred_label,
        confidence=confidence,
        filename=filename,
        advice=advice,
        products=products,
        product_images=product_images,
        error=error,
        not_a_crop=not_a_crop
    )

@app.route("/uploads/<filename>")
def uploaded_file(filename):
    return send_from_directory(app.config["UPLOAD_FOLDER"], filename)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
!mkdir -p templates
!mkdir -p static

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en" id="html-root">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>PlantGuard - Plant Disease Detection</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600&family=Fraunces:ital,wght@0,600;1,400&display=swap" rel="stylesheet">
  <link rel="stylesheet" href="{{ url_for('static', filename='styles.css') }}">
</head>
<body>

<!-- HEADER -->
<header>
  <div class="header-inner">
    <a href="/" class="logo">
      <span class="logo-icon">🌿</span>
      <span class="logo-text">PlantGuard</span>
    </a>
    <nav>
      <a href="/" class="nav-link active" data-i18n="nav_home">Home</a>
      <a href="#detect" class="nav-link" data-i18n="nav_detect">Detect</a>
      <a href="#library" class="nav-link" data-i18n="nav_library">Library</a>
      <a href="#prevention" class="nav-link" data-i18n="nav_prevention">Prevention</a>
    </nav>

    <!-- Language Selector -->
    <div class="lang-wrap">
      <button class="lang-btn" onclick="toggleLangMenu()" id="langBtn">🌐 EN ▾</button>
      <div class="lang-menu" id="langMenu">
        <button onclick="setLang('en')">🇬🇧 English</button>
        <button onclick="setLang('te')">🇮🇳 తెలుగు</button>
        <button onclick="setLang('hi')">🇮🇳 हिन्दी</button>
        <button onclick="setLang('ta')">🇮🇳 தமிழ்</button>
        <button onclick="setLang('kn')">🇮🇳 ಕನ್ನಡ</button>
        <button onclick="setLang('ml')">🇮🇳 മലയാളം</button>
      </div>
    </div>

    <button class="nav-cta" onclick="document.getElementById('detect').scrollIntoView({behavior:'smooth'})" data-i18n="btn_try">Try Now</button>
  </div>
</header>

<!-- HERO -->
<section class="hero">
  <div class="hero-content">
    <p class="hero-tag" data-i18n="hero_tag">AI-Powered Agriculture</p>
    <h1 class="hero-title" data-i18n="hero_title">Detect Plant Diseases<br><em>Instantly</em></h1>
    <p class="hero-sub" data-i18n="hero_sub">Upload a leaf photo or use your camera. Our ViT + LLM model identifies diseases and gives expert prevention advice in seconds.</p>
    <a href="#detect" class="btn-primary" data-i18n="btn_start">Start Detection</a>
  </div>
  <div class="hero-visual">
    <div class="hero-badge">
      <span class="badge-icon">🧠</span>
      <span>ViT + Mistral-7B</span>
    </div>
    <div class="leaf-ring"></div>
    <div class="leaf-emoji">🍃</div>
  </div>
</section>

<!-- DETECT SECTION -->
<section id="detect" class="detect-section">
  <div class="section-label" data-i18n="sec_detection">Detection</div>
  <h2 class="section-title" data-i18n="sec_analyze">Analyze Your Plant</h2>
  <p class="section-sub" data-i18n="sec_analyze_sub">Choose an image source below to identify diseases in your plant leaves.</p>

  {% if error %}
  <div class="error-banner">⚠️ {{ error }}</div>
  {% endif %}

  {% if not_a_crop %}
  <div class="not-crop-card">

    <!-- Icon + Title -->
    <div class="not-crop-header">
      <span class="not-crop-icon">🚫</span>
      <div>
        <h3 class="not-crop-title">This does not look like a crop image</h3>
        <p class="not-crop-subtitle">Our ViT-Base model was trained only on plant leaf images and could not recognise this as a valid crop or leaf photo.</p>
      </div>
    </div>

    <!-- Uploaded image preview + confidence stats -->
    <div class="not-crop-body">
      {% if filename %}
      <div class="not-crop-preview-wrap">
        <p class="not-crop-label">Uploaded Image</p>
        <img src="{{ url_for('uploaded_file', filename=filename) }}" class="not-crop-preview" alt="Uploaded image">
      </div>
      {% endif %}

      <div class="not-crop-info">
        <p class="not-crop-label">Why was it rejected?</p>

        {% if top1_conf is defined and top1_conf %}
        <div class="not-crop-stat">
          <span class="stat-name">Model Confidence</span>
          <span class="stat-bar-wrap">
            <span class="stat-bar" style="width:{{ top1_conf }}%;background:{% if top1_conf < 40 %}#e74c3c{% else %}#f39c12{% endif %};"></span>
          </span>
          <span class="stat-val {% if top1_conf < 40 %}stat-low{% else %}stat-mid{% endif %}">{{ top1_conf }}%</span>
        </div>
        <p class="stat-note">Minimum required: <strong>40%</strong> — A real leaf image typically scores above 70%.</p>
        {% endif %}

        {% if entropy is defined and entropy %}
        <div class="not-crop-stat" style="margin-top:10px;">
          <span class="stat-name">Prediction Entropy</span>
          <span class="stat-bar-wrap">
            <span class="stat-bar" style="width:{{ [entropy / 5.25 * 100, 100]|min }}%;background:{% if entropy > 3.2 %}#e74c3c{% else %}#27ae60{% endif %};"></span>
          </span>
          <span class="stat-val {% if entropy > 3.2 %}stat-low{% else %}stat-ok{% endif %}">{{ entropy }}</span>
        </div>
        <p class="stat-note">Maximum allowed: <strong>3.2</strong> — High entropy means the model is uncertain and the image is unfamiliar.</p>
        {% endif %}

        <div class="not-crop-tips">
          <p class="tips-title">✅ Tips for a valid image:</p>
          <ul>
            <li>📸 Take a <strong>close-up photo</strong> of the affected leaf</li>
            <li>🌿 Make sure the <strong>leaf fills most of the frame</strong></li>
            <li>💡 Use <strong>good lighting</strong> — avoid dark or blurry photos</li>
            <li>🚫 Do <strong>not</strong> upload photos of people, animals, objects, or food</li>
            <li>🌾 Supported crops: Tomato, Potato, Apple, Corn, Grape, Peach, Bell Pepper, Cherry, Strawberry</li>
          </ul>
        </div>
      </div>
    </div>

    <a href="#detect" class="btn-primary not-crop-btn"
       onclick="document.getElementById('detect').scrollIntoView({behavior:'smooth'});return false;">
      ↩ Try Again with a Leaf Image
    </a>
  </div>
  {% endif %}

  <div class="detect-card">
    <div class="tabs">
      <button class="tab active" onclick="switchTab('upload', this)" data-i18n="tab_upload">📁 Upload Image</button>
      <button class="tab" onclick="switchTab('camera', this)" data-i18n="tab_camera">📷 Use Camera</button>
    </div>

    <!-- Upload Panel -->
    <div id="tab-upload" class="tab-panel active">
      <form method="POST" enctype="multipart/form-data" id="uploadForm">
        <label class="drop-zone" id="dropZone">
          <input type="file" name="file" id="fileInput" accept="image/*" onchange="previewFile(this)">
          <div class="drop-content" id="dropContent">
            <span class="drop-icon">🖼️</span>
            <p class="drop-text" data-i18n="drop_text">Drag and drop or <u>browse</u></p>
            <p class="drop-hint" data-i18n="drop_hint">PNG, JPG, JPEG supported</p>
          </div>
          <img id="previewImg" class="preview-img" src="" alt="" style="display:none;">
        </label>
        <button type="submit" class="btn-primary full-width" id="uploadBtn" style="display:none;" data-i18n="btn_detect">🔍 Detect Disease</button>
      </form>
    </div>

    <!-- Camera Panel -->
    <div id="tab-camera" class="tab-panel">
      <div class="camera-wrapper">
        <video id="video" autoplay playsinline></video>
        <canvas id="canvas" style="display:none;"></canvas>
      </div>
      <div class="camera-actions">
        <button class="btn-secondary" id="startBtn" onclick="startCamera()" data-i18n="btn_start_cam">▶ Start Camera</button>
        <button class="btn-primary" id="captureBtn" onclick="capturePhoto()" data-i18n="btn_capture">📸 Capture and Detect</button>
      </div>
      <form id="cameraForm" method="POST" action="/">
        <input type="hidden" name="image_data" id="image_data">
      </form>
    </div>
  </div>
</section>

<!-- RESULTS -->
{% if pred_label %}
<section class="results-section">
  <div class="section-label" data-i18n="sec_results">Results</div>
  <h2 class="section-title" data-i18n="sec_report">Detection Report</h2>

  <!-- Translate advice button -->
  <div class="translate-bar">
    <span data-i18n="translate_label">Translate advice to:</span>
    <div class="translate-btns">
      <button class="tr-btn active" onclick="translateAdvice('en', this)">English</button>
      <button class="tr-btn" onclick="translateAdvice('te', this)">తెలుగు</button>
      <button class="tr-btn" onclick="translateAdvice('hi', this)">हिन्दी</button>
      <button class="tr-btn" onclick="translateAdvice('ta', this)">தமிழ்</button>
      <button class="tr-btn" onclick="translateAdvice('kn', this)">ಕನ್ನಡ</button>
      <button class="tr-btn" onclick="translateAdvice('ml', this)">മലയാളം</button>
    </div>
  </div>

  <div class="results-grid">
    {% if filename %}
    <div class="result-card image-card">
      <p class="card-label" data-i18n="card_image">Analyzed Image</p>
      <img src="{{ url_for('uploaded_file', filename=filename) }}" alt="Leaf" class="result-img">
    </div>
    {% endif %}

    <div class="result-card diagnosis-card">
      <p class="card-label" data-i18n="card_diagnosis">Diagnosis</p>
      <h3 class="disease-name">{{ pred_label }}</h3>
      {% if confidence %}
      <div class="conf-header">
        <span data-i18n="conf_label">Confidence</span>
        <span class="conf-value">{{ "%.1f"|format(confidence) }}%</span>
      </div>
      <div class="confidence-bar">
        <div class="confidence-fill" style="width: {{ confidence }}%;"></div>
      </div>
      <div class="conf-tag {% if confidence >= 80 %}high{% elif confidence >= 50 %}medium{% else %}low{% endif %}">
        {% if confidence >= 80 %}<span data-i18n="conf_high">High Confidence</span>
        {% elif confidence >= 50 %}<span data-i18n="conf_med">Moderate Confidence</span>
        {% else %}<span data-i18n="conf_low">Low Confidence</span>{% endif %}
      </div>
      {% endif %}
    </div>
  </div>

  {% if advice %}
  <div id="prevention" class="advice-card">
    <div class="advice-header">
      <span class="advice-icon">🌱</span>
      <h3 data-i18n="advice_title">Prevention and Treatment Advice</h3>
      <span class="translate-status" id="translateStatus"></span>
    </div>
    <div class="advice-body">
      <pre class="advice-text" id="adviceText">{{ advice }}</pre>
    </div>
  </div>
  {% endif %}

  {% if products %}
  <div id="library" class="products-wrap">
    <div class="section-label" data-i18n="sec_recommended">Recommended</div>
    <h2 class="section-title" data-i18n="sec_products">Treatment Products</h2>
    <div class="product-grid">
      {% for product in products %}
      <div class="product-card">
        <div class="product-img-wrap">
          <img src="{{ url_for('static', filename='products/' + product_images.get(product, 'neem_oil.jpg')) }}" alt="{{ product }}" class="product-img">
        </div>
        <div class="product-info">
          <p class="product-name">{{ product }}</p>
          <div class="product-actions">
            <a href="https://bighaat.com/s?k={{ product }}" target="_blank" class="btn-buy" data-i18n="btn_buy">🛒 Buy</a>
            <a href="https://www.google.com/search?q={{ product }}+pesticide+fungicide" target="_blank" class="btn-view" data-i18n="btn_info">🔎 Info</a>
          </div>
        </div>
      </div>
      {% endfor %}
    </div>
  </div>
  {% endif %}

</section>
{% endif %}

<!-- HOW IT WORKS -->
<section class="how-section">
  <div class="section-label" data-i18n="sec_process">Process</div>
  <h2 class="section-title" data-i18n="sec_how">How It Works</h2>
  <div class="steps-grid">
    <div class="step-card">
      <div class="step-num">01</div>
      <div class="step-icon">📸</div>
      <h4 data-i18n="step1_title">Capture or Upload</h4>
      <p data-i18n="step1_desc">Take a photo of the plant leaf or upload an existing image from your device.</p>
    </div>
    <div class="step-card">
      <div class="step-num">02</div>
      <div class="step-icon">🧠</div>
      <h4 data-i18n="step2_title">ViT Analysis</h4>
      <p data-i18n="step2_desc">Our Vision Transformer model classifies the disease from the leaf image with high accuracy.</p>
    </div>
    <div class="step-card">
      <div class="step-num">03</div>
      <div class="step-icon">💡</div>
      <h4 data-i18n="step3_title">LLM Advice</h4>
      <p data-i18n="step3_desc">Mistral-7B generates expert prevention tips, treatments, and product recommendations.</p>
    </div>
  </div>
</section>

<!-- ABOUT MODAL -->
<div id="modal-about" class="modal-overlay" onclick="closeModal('about')">
  <div class="modal-box" onclick="event.stopPropagation()">
    <button class="modal-close" onclick="closeModal('about')">✕</button>
    <div class="modal-icon">🌿</div>
    <h2 class="modal-title">About PlantGuard</h2>
    <p class="modal-lead">An AI-powered plant disease detection system built to help farmers and agricultural professionals identify and treat crop diseases instantly.</p>
    <div class="modal-divider"></div>
    <div class="about-grid">
      <div class="about-block"><span class="about-icon">🧠</span><h4>Vision Transformer (ViT)</h4><p>Fine-tuned <strong>vit_base_patch16_224</strong> trained on PlantVillage — 38 disease classes, 99% accuracy.</p></div>
      <div class="about-block"><span class="about-icon">💬</span><h4>Mistral-7B LLM</h4><p><strong>Mistral-7B-Instruct</strong> generates expert advice: causes, symptoms, organic and chemical treatments, product names.</p></div>
      <div class="about-block"><span class="about-icon">📸</span><h4>Dual Input Support</h4><p>Upload an existing leaf image or capture one live with your device camera for instant analysis.</p></div>
      <div class="about-block"><span class="about-icon">🌐</span><h4>6-Language Support</h4><p>Prevention advice can be translated into Telugu, Hindi, Tamil, Kannada, and Malayalam for regional farmers.</p></div>
    </div>
    <div class="modal-divider"></div>
    <p class="modal-footer-note">Built with PyTorch · Flask · HuggingFace · Google Colab · Sasi Institute of Technology and Engineering</p>
  </div>
</div>

<!-- CONTACT MODAL -->
<div id="modal-contact" class="modal-overlay" onclick="closeModal('contact')">
  <div class="modal-box modal-box--sm" onclick="event.stopPropagation()">
    <button class="modal-close" onclick="closeModal('contact')">✕</button>
    <div class="modal-icon">📬</div>
    <h2 class="modal-title">Contact Us</h2>
    <p class="modal-lead">Have questions or want to collaborate? Reach out to the project team.</p>
    <div class="modal-divider"></div>
    <div class="contact-row"><span class="contact-icon">👤</span><div><p class="contact-label">Project Developer</p><p class="contact-value">Yaswanth Kodavali</p></div></div>
    <div class="contact-row"><span class="contact-icon">🏫</span><div><p class="contact-label">Institution</p><p class="contact-value">Sasi Institute of Technology and Engineering</p></div></div>
    <div class="contact-row"><span class="contact-icon">✉️</span><div><p class="contact-label">Email</p><a href="mailto:yaswanth.kodavali@sasi.ac.in" class="contact-email">yaswanth.kodavali@sasi.ac.in</a></div></div>
    <div class="modal-divider"></div>
    <a href="mailto:yaswanth.kodavali@sasi.ac.in" class="btn-primary full-width" style="margin-top:0;">Send an Email</a>
  </div>
</div>

<!-- FOOTER -->
<footer>
  <div class="footer-inner">
    <div class="footer-brand">
      <span class="logo-icon">🌿</span>
      <span class="logo-text">PlantGuard</span>
      <p class="footer-tagline" data-i18n="footer_tag">Empowering farmers with AI</p>
    </div>
    <div class="footer-links">
      <a href="#" onclick="openModal('about'); return false;" data-i18n="footer_about">About Us</a>
      <a href="#" onclick="openModal('contact'); return false;" data-i18n="footer_contact">Contact</a>
    </div>
    <p class="footer-copy">© 2026 PlantGuard · ViT + Mistral-7B · Sasi Institute of Technology and Engineering</p>
  </div>
</footer>

<script>
// ── I18N Translations ─────────────────────────────────────────────────────
var TRANSLATIONS = {
  en: {
    nav_home:'Home', nav_detect:'Detect', nav_library:'Library', nav_prevention:'Prevention',
    btn_try:'Try Now', hero_tag:'AI-Powered Agriculture',
    hero_title:'Detect Plant Diseases<br><em>Instantly</em>',
    hero_sub:'Upload a leaf photo or use your camera. Our ViT + LLM model identifies diseases and gives expert prevention advice in seconds.',
    btn_start:'Start Detection', sec_detection:'Detection', sec_analyze:'Analyze Your Plant',
    sec_analyze_sub:'Choose an image source below to identify diseases in your plant leaves.',
    tab_upload:'📁 Upload Image', tab_camera:'📷 Use Camera',
    drop_text:'Drag and drop or <u>browse</u>', drop_hint:'PNG, JPG, JPEG supported',
    btn_detect:'🔍 Detect Disease', btn_start_cam:'▶ Start Camera', btn_capture:'📸 Capture and Detect',
    sec_results:'Results', sec_report:'Detection Report', translate_label:'Translate advice to:',
    card_image:'Analyzed Image', card_diagnosis:'Diagnosis', conf_label:'Confidence',
    conf_high:'High Confidence', conf_med:'Moderate Confidence', conf_low:'Low Confidence',
    advice_title:'Prevention and Treatment Advice',
    sec_recommended:'Recommended', sec_products:'Treatment Products',
    btn_buy:'🛒 Buy', btn_info:'🔎 Info',
    sec_process:'Process', sec_how:'How It Works',
    step1_title:'Capture or Upload', step1_desc:'Take a photo of the plant leaf or upload an existing image from your device.',
    step2_title:'ViT Analysis', step2_desc:'Our Vision Transformer model classifies the disease from the leaf image with high accuracy.',
    step3_title:'LLM Advice', step3_desc:'Mistral-7B generates expert prevention tips, treatments, and product recommendations.',
    footer_tag:'Empowering farmers with AI', footer_about:'About Us', footer_contact:'Contact'
  },
  te: {
    nav_home:'హోమ్', nav_detect:'గుర్తింపు', nav_library:'లైబ్రరీ', nav_prevention:'నివారణ',
    btn_try:'ప్రయత్నించండి', hero_tag:'AI ఆధారిత వ్యవసాయం',
    hero_title:'మొక్కల వ్యాధులను<br><em>తక్షణమే</em> గుర్తించండి',
    hero_sub:'ఆకు ఫోటో అప్లోడ్ చేయండి లేదా కెమెరా ఉపయోగించండి. మా ViT + LLM మోడల్ వ్యాధులను గుర్తిస్తుంది.',
    btn_start:'గుర్తింపు ప్రారంభించండి', sec_detection:'గుర్తింపు', sec_analyze:'మీ మొక్కను విశ్లేషించండి',
    sec_analyze_sub:'మీ మొక్క ఆకులలో వ్యాధులను గుర్తించడానికి క్రింది ఇమేజ్ సోర్స్ ఎంచుకోండి.',
    tab_upload:'📁 చిత్రం అప్లోడ్', tab_camera:'📷 కెమెరా వాడండి',
    drop_text:'లాగి వదలండి లేదా <u>బ్రౌజ్</u> చేయండి', drop_hint:'PNG, JPG, JPEG మద్దతు',
    btn_detect:'🔍 వ్యాధి గుర్తించండి', btn_start_cam:'▶ కెమెరా ప్రారంభించండి', btn_capture:'📸 క్యాప్చర్ చేసి గుర్తించండి',
    sec_results:'ఫలితాలు', sec_report:'గుర్తింపు నివేదిక', translate_label:'సలహాను అనువదించండి:',
    card_image:'విశ్లేషించిన చిత్రం', card_diagnosis:'నిర్ధారణ', conf_label:'నమ్మకం',
    conf_high:'అధిక నమ్మకం', conf_med:'మధ్యస్థ నమ్మకం', conf_low:'తక్కువ నమ్మకం',
    advice_title:'నివారణ మరియు చికిత్స సలహా',
    sec_recommended:'సిఫార్సు చేయబడినవి', sec_products:'చికిత్స ఉత్పత్తులు',
    btn_buy:'🛒 కొనండి', btn_info:'🔎 సమాచారం',
    sec_process:'ప్రక్రియ', sec_how:'ఇది ఎలా పని చేస్తుంది',
    step1_title:'క్యాప్చర్ లేదా అప్లోడ్', step1_desc:'మొక్క ఆకు ఫోటో తీయండి లేదా పరికరం నుండి చిత్రం అప్లోడ్ చేయండి.',
    step2_title:'ViT విశ్లేషణ', step2_desc:'మా Vision Transformer మోడల్ ఆకు చిత్రం నుండి వ్యాధిని గుర్తిస్తుంది.',
    step3_title:'LLM సలహా', step3_desc:'Mistral-7B నిపుణుల నివారణ చిట్కాలు మరియు ఉత్పత్తి సిఫార్సులు అందిస్తుంది.',
    footer_tag:'AI తో రైతులను శక్తివంతం చేయడం', footer_about:'మా గురించి', footer_contact:'సంప్రదించండి'
  },
  hi: {
    nav_home:'होम', nav_detect:'पहचानें', nav_library:'लाइब्रेरी', nav_prevention:'रोकथाम',
    btn_try:'अभी आज़माएं', hero_tag:'AI-संचालित कृषि',
    hero_title:'पौधों की बीमारियाँ<br><em>तुरंत</em> पहचानें',
    hero_sub:'पत्ती की फ़ोटो अपलोड करें या कैमरा उपयोग करें। हमारा ViT + LLM मॉडल बीमारी पहचानकर विशेषज्ञ सलाह देता है।',
    btn_start:'पहचान शुरू करें', sec_detection:'पहचान', sec_analyze:'अपने पौधे का विश्लेषण करें',
    sec_analyze_sub:'नीचे इमेज स्रोत चुनें और अपने पौधे की पत्तियों में बीमारी पहचानें।',
    tab_upload:'📁 छवि अपलोड करें', tab_camera:'📷 कैमरा उपयोग करें',
    drop_text:'खींचें और छोड़ें या <u>ब्राउज़ करें</u>', drop_hint:'PNG, JPG, JPEG समर्थित',
    btn_detect:'🔍 बीमारी पहचानें', btn_start_cam:'▶ कैमरा शुरू करें', btn_capture:'📸 कैप्चर करें और पहचानें',
    sec_results:'परिणाम', sec_report:'पहचान रिपोर्ट', translate_label:'सलाह अनुवाद करें:',
    card_image:'विश्लेषित छवि', card_diagnosis:'निदान', conf_label:'विश्वास',
    conf_high:'उच्च विश्वास', conf_med:'मध्यम विश्वास', conf_low:'कम विश्वास',
    advice_title:'रोकथाम और उपचार सलाह',
    sec_recommended:'अनुशंसित', sec_products:'उपचार उत्पाद',
    btn_buy:'🛒 खरीदें', btn_info:'🔎 जानकारी',
    sec_process:'प्रक्रिया', sec_how:'यह कैसे काम करता है',
    step1_title:'कैप्चर या अपलोड', step1_desc:'पौधे की पत्ती की फ़ोटो लें या डिवाइस से छवि अपलोड करें।',
    step2_title:'ViT विश्लेषण', step2_desc:'हमारा Vision Transformer मॉडल पत्ती की छवि से बीमारी वर्गीकृत करता है।',
    step3_title:'LLM सलाह', step3_desc:'Mistral-7B विशेषज्ञ रोकथाम सुझाव और उत्पाद अनुशंसाएं देता है।',
    footer_tag:'AI से किसानों को सशक्त बनाना', footer_about:'हमारे बारे में', footer_contact:'संपर्क करें'
  },
  ta: {
    nav_home:'முகப்பு', nav_detect:'கண்டறிதல்', nav_library:'நூலகம்', nav_prevention:'தடுப்பு',
    btn_try:'இப்போது முயற்சி', hero_tag:'AI-இயக்கப்படும் விவசாயம்',
    hero_title:'தாவர நோய்களை<br><em>உடனடியாக</em> கண்டறியுங்கள்',
    hero_sub:'இலை புகைப்படம் பதிவேற்றவும் அல்லது கேமரா பயன்படுத்தவும். எங்கள் ViT + LLM மாதிரி நோய்களை கண்டறிகிறது.',
    btn_start:'கண்டறிதல் தொடங்கு', sec_detection:'கண்டறிதல்', sec_analyze:'உங்கள் தாவரத்தை பகுப்பாய்வு செய்யுங்கள்',
    sec_analyze_sub:'உங்கள் தாவர இலைகளில் நோய்களை கண்டறிய படம் தேர்ந்தெடுக்கவும்.',
    tab_upload:'📁 படம் பதிவேற்று', tab_camera:'📷 கேமரா பயன்படுத்து',
    drop_text:'இழுத்து விடவும் அல்லது <u>உலாவு</u>', drop_hint:'PNG, JPG, JPEG ஆதரவு',
    btn_detect:'🔍 நோய் கண்டறி', btn_start_cam:'▶ கேமரா தொடங்கு', btn_capture:'📸 படம் எடுத்து கண்டறி',
    sec_results:'முடிவுகள்', sec_report:'கண்டறிதல் அறிக்கை', translate_label:'ஆலோசனையை மொழிபெயர்க்கவும்:',
    card_image:'பகுப்பாய்வு செய்யப்பட்ட படம்', card_diagnosis:'நோயறிதல்', conf_label:'நம்பிக்கை',
    conf_high:'அதிக நம்பிக்கை', conf_med:'மிதமான நம்பிக்கை', conf_low:'குறைந்த நம்பிக்கை',
    advice_title:'தடுப்பு மற்றும் சிகிச்சை ஆலோசனை',
    sec_recommended:'பரிந்துரைக்கப்பட்டவை', sec_products:'சிகிச்சை தயாரிப்புகள்',
    btn_buy:'🛒 வாங்கு', btn_info:'🔎 தகவல்',
    sec_process:'செயல்முறை', sec_how:'இது எவ்வாறு செயல்படுகிறது',
    step1_title:'படம் எடு அல்லது பதிவேற்று', step1_desc:'தாவர இலையின் புகைப்படம் எடுக்கவும் அல்லது சாதனத்திலிருந்து படம் பதிவேற்றவும்.',
    step2_title:'ViT பகுப்பாய்வு', step2_desc:'எங்கள் Vision Transformer மாதிரி இலை படத்திலிருந்து நோயை வகைப்படுத்துகிறது.',
    step3_title:'LLM ஆலோசனை', step3_desc:'Mistral-7B நிபுணர் தடுப்பு குறிப்புகள் மற்றும் தயாரிப்பு பரிந்துரைகளை வழங்குகிறது.',
    footer_tag:'AI மூலம் விவசாயிகளை வலுப்படுத்துதல்', footer_about:'எங்களைப் பற்றி', footer_contact:'தொடர்பு கொள்ளுங்கள்'
  },
  kn: {
    nav_home:'ಮುಖಪುಟ', nav_detect:'ಪತ್ತೆ', nav_library:'ಗ್ರಂಥಾಲಯ', nav_prevention:'ತಡೆಗಟ್ಟುವಿಕೆ',
    btn_try:'ಈಗ ಪ್ರಯತ್ನಿಸಿ', hero_tag:'AI-ಚಾಲಿತ ಕೃಷಿ',
    hero_title:'ಸಸ್ಯ ರೋಗಗಳನ್ನು<br><em>ತಕ್ಷಣ</em> ಪತ್ತೆ ಮಾಡಿ',
    hero_sub:'ಎಲೆಯ ಫೋಟೋ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ ಅಥವಾ ಕ್ಯಾಮೆರಾ ಬಳಸಿ. ನಮ್ಮ ViT + LLM ಮಾದರಿ ರೋಗಗಳನ್ನು ಗುರುತಿಸುತ್ತದೆ.',
    btn_start:'ಪತ್ತೆ ಪ್ರಾರಂಭಿಸಿ', sec_detection:'ಪತ್ತೆ', sec_analyze:'ನಿಮ್ಮ ಸಸ್ಯವನ್ನು ವಿಶ್ಲೇಷಿಸಿ',
    sec_analyze_sub:'ನಿಮ್ಮ ಸಸ್ಯದ ಎಲೆಗಳಲ್ಲಿ ರೋಗಗಳನ್ನು ಪತ್ತೆ ಮಾಡಲು ಚಿತ್ರ ಮೂಲ ಆಯ್ಕೆಮಾಡಿ.',
    tab_upload:'📁 ಚಿತ್ರ ಅಪ್‌ಲೋಡ್', tab_camera:'📷 ಕ್ಯಾಮೆರಾ ಬಳಸಿ',
    drop_text:'ಎಳೆದು ಬಿಡಿ ಅಥವಾ <u>ಬ್ರೌಸ್ ಮಾಡಿ</u>', drop_hint:'PNG, JPG, JPEG ಬೆಂಬಲಿತ',
    btn_detect:'🔍 ರೋಗ ಪತ್ತೆ ಮಾಡಿ', btn_start_cam:'▶ ಕ್ಯಾಮೆರಾ ಪ್ರಾರಂಭಿಸಿ', btn_capture:'📸 ಕ್ಯಾಪ್ಚರ್ ಮಾಡಿ ಮತ್ತು ಪತ್ತೆ ಮಾಡಿ',
    sec_results:'ಫಲಿತಾಂಶಗಳು', sec_report:'ಪತ್ತೆ ವರದಿ', translate_label:'ಸಲಹೆ ಅನುವಾದಿಸಿ:',
    card_image:'ವಿಶ್ಲೇಷಿಸಿದ ಚಿತ್ರ', card_diagnosis:'ರೋಗ ನಿರ್ಣಯ', conf_label:'ವಿಶ್ವಾಸ',
    conf_high:'ಹೆಚ್ಚಿನ ವಿಶ್ವಾಸ', conf_med:'ಮಧ್ಯಮ ವಿಶ್ವಾಸ', conf_low:'ಕಡಿಮೆ ವಿಶ್ವಾಸ',
    advice_title:'ತಡೆಗಟ್ಟುವಿಕೆ ಮತ್ತು ಚಿಕಿತ್ಸಾ ಸಲಹೆ',
    sec_recommended:'ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ', sec_products:'ಚಿಕಿತ್ಸಾ ಉತ್ಪನ್ನಗಳು',
    btn_buy:'🛒 ಖರೀದಿಸಿ', btn_info:'🔎 ಮಾಹಿತಿ',
    sec_process:'ಪ್ರಕ್ರಿಯೆ', sec_how:'ಇದು ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ',
    step1_title:'ಕ್ಯಾಪ್ಚರ್ ಅಥವಾ ಅಪ್‌ಲೋಡ್', step1_desc:'ಸಸ್ಯದ ಎಲೆಯ ಫೋಟೋ ತೆಗೆಯಿರಿ ಅಥವಾ ಸಾಧನದಿಂದ ಚಿತ್ರ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ.',
    step2_title:'ViT ವಿಶ್ಲೇಷಣೆ', step2_desc:'ನಮ್ಮ Vision Transformer ಮಾದರಿ ಎಲೆ ಚಿತ್ರದಿಂದ ರೋಗ ವರ್ಗೀಕರಿಸುತ್ತದೆ.',
    step3_title:'LLM ಸಲಹೆ', step3_desc:'Mistral-7B ತಜ್ಞ ತಡೆಗಟ್ಟುವಿಕೆ ಸಲಹೆಗಳು ಮತ್ತು ಉತ್ಪನ್ನ ಶಿಫಾರಸುಗಳನ್ನು ನೀಡುತ್ತದೆ.',
    footer_tag:'AI ಮೂಲಕ ರೈತರನ್ನು ಸಶಕ್ತಗೊಳಿಸಲಾಗುತ್ತಿದೆ', footer_about:'ನಮ್ಮ ಬಗ್ಗೆ', footer_contact:'ಸಂಪರ್ಕಿಸಿ'
  },
  ml: {
    nav_home:'ഹോം', nav_detect:'കണ്ടെത്തൽ', nav_library:'ലൈബ്രറി', nav_prevention:'പ്രതിരോധം',
    btn_try:'ഇപ്പോൾ ശ്രമിക്കൂ', hero_tag:'AI-ചാലിത കൃഷി',
    hero_title:'സസ്യ രോഗങ്ങൾ<br><em>ഉടൻ</em> കണ്ടെത്തുക',
    hero_sub:'ഇല ഫോട്ടോ അപ്‌ലോഡ് ചെയ്യുക അല്ലെങ്കിൽ ക്യാമറ ഉപയോഗിക്കുക. ഞങ്ങളുടെ ViT + LLM മോഡൽ രോഗങ്ങൾ തിരിച്ചറിയുന്നു.',
    btn_start:'കണ്ടെത്തൽ ആരംഭിക്കുക', sec_detection:'കണ്ടെത്തൽ', sec_analyze:'നിങ്ങളുടെ സസ്യം വിശകലനം ചെയ്യുക',
    sec_analyze_sub:'സസ്യ ഇലകളിലെ രോഗങ്ങൾ കണ്ടെത്താൻ ചിത്ര ഉറവിടം തിരഞ്ഞെടുക്കുക.',
    tab_upload:'📁 ചിത്രം അപ്‌ലോഡ്', tab_camera:'📷 ക്യാമറ ഉപയോഗിക്കുക',
    drop_text:'വലിച്ചിടുക അല്ലെങ്കിൽ <u>ബ്രൗസ് ചെയ്യുക</u>', drop_hint:'PNG, JPG, JPEG പിന്തുണ',
    btn_detect:'🔍 രോഗം കണ്ടെത്തുക', btn_start_cam:'▶ ക്യാമറ ആരംഭിക്കുക', btn_capture:'📸 ക്യാപ്ചർ ചെയ്ത് കണ്ടെത്തുക',
    sec_results:'ഫലങ്ങൾ', sec_report:'കണ്ടെത്തൽ റിപ്പോർട്ട്', translate_label:'ഉപദേശം വിവർത്തനം ചെയ്യുക:',
    card_image:'വിശകലനം ചെയ്ത ചിത്രം', card_diagnosis:'രോഗനിർണ്ണയം', conf_label:'ആത്മവിശ്വാസം',
    conf_high:'ഉയർന്ന ആത്മവിശ്വാസം', conf_med:'മിതമായ ആത്മവിശ്വാസം', conf_low:'കുറഞ്ഞ ആത്മവിശ്വാസം',
    advice_title:'പ്രതിരോധ, ചികിത്സ ഉപദേശം',
    sec_recommended:'ശുപാർശ ചെയ്തത്', sec_products:'ചികിത്സ ഉൽപ്പന്നങ്ങൾ',
    btn_buy:'🛒 വാങ്ങുക', btn_info:'🔎 വിവരം',
    sec_process:'പ്രക്രിയ', sec_how:'ഇത് എങ്ങനെ പ്രവർത്തിക്കുന്നു',
    step1_title:'ക്യാപ്ചർ അല്ലെങ്കിൽ അപ്‌ലോഡ്', step1_desc:'സസ്യ ഇലയുടെ ഫോട്ടോ എടുക്കുക അല്ലെങ്കിൽ ഉപകരണത്തിൽ നിന്ന് ചിത്രം അപ്‌ലോഡ് ചെയ്യുക.',
    step2_title:'ViT വിശകലനം', step2_desc:'ഞങ്ങളുടെ Vision Transformer മോഡൽ ഇല ചിത്രത്തിൽ നിന്ന് രോഗം തരംതിരിക്കുന്നു.',
    step3_title:'LLM ഉപദേശം', step3_desc:'Mistral-7B വിദഗ്ധ പ്രതിരോധ നുറുങ്ങുകളും ഉൽപ്പന്ന ശുപാർശകളും നൽകുന്നു.',
    footer_tag:'AI ഉപയോഗിച്ച് കർഷകരെ ശക്തിപ്പെടുത്തുന്നു', footer_about:'ഞങ്ങളെക്കുറിച്ച്', footer_contact:'ബന്ധപ്പെടുക'
  }
};

var currentLang = 'en';
var originalAdvice = null;

function setLang(lang) {
  currentLang = lang;
  var tr = TRANSLATIONS[lang] || TRANSLATIONS['en'];
  var langNames = {en:'EN', te:'TE', hi:'HI', ta:'TA', kn:'KN', ml:'ML'};
  document.getElementById('langBtn').textContent = '🌐 ' + langNames[lang] + ' ▾';
  document.getElementById('langMenu').classList.remove('open');

  document.querySelectorAll('[data-i18n]').forEach(function(el) {
    var key = el.getAttribute('data-i18n');
    if (tr[key] !== undefined) {
      el.innerHTML = tr[key];
    }
  });
}

function toggleLangMenu() {
  document.getElementById('langMenu').classList.toggle('open');
}
document.addEventListener('click', function(e) {
  var w = document.querySelector('.lang-wrap');
  if (w && !w.contains(e.target)) {
    document.getElementById('langMenu').classList.remove('open');
  }
});

// ── Advice Translator (Google Translate API free endpoint) ────────────────
var LANG_CODES = {en:'en', te:'te', hi:'hi', ta:'ta', kn:'kn', ml:'ml'};

function translateAdvice(targetLang, btn) {
  var adviceEl = document.getElementById('adviceText');
  var statusEl = document.getElementById('translateStatus');
  if (!adviceEl) return;

  // Save original English text
  if (!originalAdvice) {
    originalAdvice = adviceEl.textContent;
  }

  // Reset to English first
  if (targetLang === 'en') {
    adviceEl.textContent = originalAdvice;
    document.querySelectorAll('.tr-btn').forEach(function(b){ b.classList.remove('active'); });
    btn.classList.add('active');
    if (statusEl) statusEl.textContent = '';
    return;
  }

  document.querySelectorAll('.tr-btn').forEach(function(b){ b.classList.remove('active'); });
  btn.classList.add('active');
  if (statusEl) statusEl.textContent = '⏳ Translating...';

  var text = originalAdvice;
  var url = 'https://translate.googleapis.com/translate_a/single?client=gtx&sl=en&tl='
            + LANG_CODES[targetLang] + '&dt=t&q=' + encodeURIComponent(text);

  fetch(url)
    .then(function(r){ return r.json(); })
    .then(function(data) {
      var translated = '';
      if (data && data[0]) {
        data[0].forEach(function(chunk) {
          if (chunk[0]) translated += chunk[0];
        });
      }
      adviceEl.textContent = translated || text;
      if (statusEl) statusEl.textContent = '✅ Translated';
      setTimeout(function(){ if(statusEl) statusEl.textContent=''; }, 2000);
    })
    .catch(function() {
      if (statusEl) statusEl.textContent = '⚠️ Translation unavailable';
      setTimeout(function(){ if(statusEl) statusEl.textContent=''; }, 3000);
    });
}

// ── Tab switching ─────────────────────────────────────────────────────────
function switchTab(tab, btn) {
  document.querySelectorAll('.tab-panel').forEach(function(p){ p.classList.remove('active'); });
  document.querySelectorAll('.tab').forEach(function(t){ t.classList.remove('active'); });
  document.getElementById('tab-' + tab).classList.add('active');
  btn.classList.add('active');
}

// ── File preview ──────────────────────────────────────────────────────────
function previewFile(input) {
  var file = input.files[0];
  if (!file) return;
  var reader = new FileReader();
  reader.onload = function(e) {
    var img = document.getElementById('previewImg');
    img.src = e.target.result;
    img.style.display = 'block';
    document.getElementById('dropContent').style.display = 'none';
    document.getElementById('uploadBtn').style.display = 'block';
  };
  reader.readAsDataURL(file);
}

// ── Drag & drop ───────────────────────────────────────────────────────────
var dz = document.getElementById('dropZone');
if (dz) {
  dz.addEventListener('dragover', function(e){ e.preventDefault(); dz.classList.add('drag-over'); });
  dz.addEventListener('dragleave', function(){ dz.classList.remove('drag-over'); });
  dz.addEventListener('drop', function(e){
    e.preventDefault(); dz.classList.remove('drag-over');
    var f = e.dataTransfer.files[0];
    if (f){ document.getElementById('fileInput').files = e.dataTransfer.files; previewFile(document.getElementById('fileInput')); }
  });
}

// ── Camera ────────────────────────────────────────────────────────────────
var stream = null;

function startCamera() {
  navigator.mediaDevices.getUserMedia({ video: { facingMode: 'environment' } })
    .then(function(s) {
      stream = s;
      var video = document.getElementById('video');
      video.srcObject = stream;
      video.play();
      document.getElementById('startBtn').textContent = 'Camera On';
    })
    .catch(function() {
      alert('Camera access denied. Please allow camera permission and try again.');
    });
}

function capturePhoto() {
  var video = document.getElementById('video');
  if (!stream) { alert('Please click Start Camera first.'); return; }
  if (video.readyState < 2) { alert('Camera is loading, please wait.'); return; }
  if (video.videoWidth === 0) { alert('Camera not ready. Please wait and try again.'); return; }

  var canvas = document.getElementById('canvas');
  var maxW = 800;
  var scale = Math.min(1, maxW / video.videoWidth);
  canvas.width  = Math.round(video.videoWidth * scale);
  canvas.height = Math.round(video.videoHeight * scale);
  canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);

  var data = canvas.toDataURL('image/jpeg', 0.75);
  if (!data || data === 'data:,') { alert('Failed to capture. Please try again.'); return; }

  document.getElementById('image_data').value = data;
  stream.getTracks().forEach(function(t){ t.stop(); });

  var btn = document.getElementById('captureBtn');
  btn.textContent = 'Processing...';
  btn.disabled = true;
  document.getElementById('cameraForm').submit();
}

// ── Modals ────────────────────────────────────────────────────────────────
function openModal(id) { document.getElementById('modal-'+id).classList.add('active'); document.body.style.overflow='hidden'; }
function closeModal(id) { document.getElementById('modal-'+id).classList.remove('active'); document.body.style.overflow=''; }
document.addEventListener('keydown', function(e){ if(e.key==='Escape'){ closeModal('about'); closeModal('contact'); } });
</script>

</body>
</html>

In [ ]:
%%writefile static/styles.css
:root {
  --green-dark:   #1a4731;
  --green-main:   #2d7a4f;
  --green-light:  #e8f5ee;
  --green-accent: #48bb78;
  --text-dark:    #1a2e1f;
  --text-mid:     #4a6358;
  --text-light:   #8fa899;
  --bg:           #f7faf8;
  --white:        #ffffff;
  --border:       #d4e8dc;
  --shadow-sm:    0 1px 4px rgba(0,0,0,0.07);
  --shadow-md:    0 4px 16px rgba(0,0,0,0.09);
  --radius:       12px;
  --radius-lg:    20px;
}

*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
html { scroll-behavior: smooth; }
body {
  font-family: 'DM Sans', sans-serif;
  background: var(--bg);
  color: var(--text-dark);
  line-height: 1.6;
  font-size: 15px;
}

/* HEADER */
header {
  position: sticky; top: 0; z-index: 100;
  background: rgba(255,255,255,0.95);
  backdrop-filter: blur(10px);
  border-bottom: 1px solid var(--border);
  padding: 0 24px;
  height: 62px;
}
.header-inner {
  max-width: 1100px; margin: auto; height: 100%;
  display: flex; align-items: center; gap: 32px;
}
.logo {
  display: flex; align-items: center; gap: 8px;
  text-decoration: none;
  font-family: 'Fraunces', serif; font-weight: 600; font-size: 18px;
  color: var(--green-dark); white-space: nowrap;
}
.logo-icon { font-size: 20px; }
nav { display: flex; gap: 4px; margin-left: auto; }
.nav-link {
  text-decoration: none; color: var(--text-mid);
  font-size: 14px; font-weight: 500;
  padding: 6px 14px; border-radius: 8px;
  transition: background 0.18s, color 0.18s;
}
.nav-link:hover, .nav-link.active { color: var(--green-main); background: var(--green-light); }
.nav-cta {
  background: var(--green-main); color: white; border: none;
  padding: 8px 18px; border-radius: 8px;
  font-size: 14px; font-weight: 600; cursor: pointer;
  transition: background 0.18s; white-space: nowrap;
}
.nav-cta:hover { background: var(--green-dark); }

/* HERO */
.hero {
  max-width: 1100px; margin: 0 auto;
  padding: 72px 24px 64px;
  display: grid; grid-template-columns: 1fr auto; gap: 48px; align-items: center;
}
.hero-tag {
  display: inline-block;
  background: var(--green-light); color: var(--green-main);
  font-size: 12px; font-weight: 600; letter-spacing: 0.06em; text-transform: uppercase;
  padding: 4px 12px; border-radius: 20px; margin-bottom: 16px;
}
.hero-title {
  font-family: 'Fraunces', serif; font-size: 46px; line-height: 1.15;
  color: var(--text-dark); margin-bottom: 18px;
}
.hero-title em { font-style: italic; color: var(--green-main); }
.hero-sub { color: var(--text-mid); font-size: 16px; max-width: 440px; margin-bottom: 28px; }

.btn-primary {
  display: inline-block; background: var(--green-main); color: white; border: none;
  padding: 12px 26px; border-radius: var(--radius);
  font-size: 15px; font-weight: 600; cursor: pointer; text-decoration: none;
  transition: background 0.18s, transform 0.12s;
}
.btn-primary:hover { background: var(--green-dark); transform: translateY(-1px); }
.btn-primary.full-width { display: block; width: 100%; text-align: center; margin-top: 16px; }

.btn-secondary {
  display: inline-block; background: var(--white); color: var(--green-main);
  border: 1.5px solid var(--border); padding: 11px 22px; border-radius: var(--radius);
  font-size: 14px; font-weight: 600; cursor: pointer; text-decoration: none;
  transition: border-color 0.18s, background 0.18s;
}
.btn-secondary:hover { border-color: var(--green-main); background: var(--green-light); }

.hero-visual {
  position: relative; width: 220px; height: 220px;
  display: flex; align-items: center; justify-content: center;
}
.leaf-ring {
  position: absolute; width: 180px; height: 180px;
  border: 2px dashed var(--green-accent); border-radius: 50%;
  animation: spin 18s linear infinite;
}
@keyframes spin { to { transform: rotate(360deg); } }
.leaf-emoji { font-size: 72px; }
.hero-badge {
  position: absolute; top: 10px; right: -20px;
  background: white; border: 1px solid var(--border); box-shadow: var(--shadow-md);
  padding: 6px 12px; border-radius: 20px;
  font-size: 12px; font-weight: 600; color: var(--green-dark);
  display: flex; align-items: center; gap: 6px;
}
.badge-icon { font-size: 14px; }

/* SECTION HELPERS */
.section-label {
  font-size: 11px; font-weight: 700; letter-spacing: 0.1em; text-transform: uppercase;
  color: var(--green-main); margin-bottom: 8px;
}
.section-title {
  font-family: 'Fraunces', serif; font-size: 30px;
  color: var(--text-dark); margin-bottom: 10px;
}
.section-sub { color: var(--text-mid); margin-bottom: 36px; max-width: 520px; }

/* DETECT SECTION */
.detect-section { max-width: 680px; margin: 0 auto; padding: 60px 24px; }
.detect-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: var(--radius-lg); box-shadow: var(--shadow-md); padding: 32px;
}
.tabs {
  display: flex; gap: 8px; margin-bottom: 24px;
  background: var(--bg); padding: 6px; border-radius: var(--radius);
}
.tab {
  flex: 1; background: transparent; border: none;
  padding: 9px 16px; border-radius: 8px;
  font-size: 14px; font-weight: 600; color: var(--text-mid); cursor: pointer;
  transition: background 0.18s, color 0.18s;
}
.tab.active { background: var(--white); color: var(--green-main); box-shadow: var(--shadow-sm); }
.tab-panel { display: none; }
.tab-panel.active { display: block; }

.drop-zone {
  display: flex; flex-direction: column; align-items: center; justify-content: center;
  border: 2px dashed var(--border); border-radius: var(--radius);
  padding: 32px 16px; cursor: pointer;
  transition: border-color 0.18s, background 0.18s;
  background: var(--bg); position: relative; min-height: 180px;
}
.drop-zone:hover, .drop-zone.drag-over { border-color: var(--green-main); background: var(--green-light); }
.drop-zone input[type="file"] {
  position: absolute; inset: 0; opacity: 0; cursor: pointer; width: 100%; height: 100%;
}
.drop-icon { font-size: 36px; margin-bottom: 10px; }
.drop-text { font-weight: 500; color: var(--text-dark); font-size: 14px; }
.drop-hint { color: var(--text-light); font-size: 12px; margin-top: 4px; }
.preview-img { max-width: 100%; max-height: 260px; border-radius: 10px; object-fit: contain; }

.camera-wrapper {
  background: #111; border-radius: var(--radius); overflow: hidden;
  aspect-ratio: 4/3; display: flex; align-items: center; justify-content: center;
  margin-bottom: 16px;
}
#video { width: 100%; height: 100%; object-fit: cover; }
.camera-actions { display: flex; gap: 12px; }

/* RESULTS */
.results-section { max-width: 900px; margin: 0 auto; padding: 48px 24px; }
.results-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 24px; }
.result-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: var(--radius-lg); padding: 24px; box-shadow: var(--shadow-sm);
}
.card-label {
  font-size: 11px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase;
  color: var(--text-light); margin-bottom: 14px;
}
.result-img { width: 100%; max-height: 220px; object-fit: cover; border-radius: var(--radius); }
.disease-name {
  font-family: 'Fraunces', serif; font-size: 22px;
  color: #c0392b; margin-bottom: 20px; line-height: 1.3;
}
.conf-header {
  display: flex; justify-content: space-between;
  font-size: 13px; color: var(--text-mid); margin-bottom: 6px; font-weight: 500;
}
.conf-value { font-weight: 700; color: var(--green-main); }
.confidence-bar {
  background: var(--border); border-radius: 20px; height: 8px; overflow: hidden; margin-bottom: 14px;
}
.confidence-fill {
  height: 100%;
  background: linear-gradient(90deg, var(--green-accent), var(--green-main));
  border-radius: 20px; transition: width 0.6s ease;
}
.conf-tag {
  display: inline-block; font-size: 12px; font-weight: 700;
  padding: 4px 12px; border-radius: 20px;
}
.conf-tag.high { background: #e8f5ee; color: #1a7340; }
.conf-tag.medium { background: #fff8e1; color: #a07030; }
.conf-tag.low { background: #fdecea; color: #c0392b; }

/* ADVICE */
.advice-card {
  background: var(--white); border: 1px solid var(--border);
  border-left: 4px solid var(--green-main);
  border-radius: var(--radius-lg); box-shadow: var(--shadow-sm);
  margin-bottom: 40px; overflow: hidden;
}
.advice-header {
  display: flex; align-items: center; gap: 10px;
  padding: 18px 24px; border-bottom: 1px solid var(--border);
  background: var(--green-light);
}
.advice-icon { font-size: 20px; }
.advice-header h3 { font-family: 'Fraunces', serif; font-size: 17px; color: var(--green-dark); }
.advice-body { padding: 24px; }
.advice-text {
  font-family: 'DM Sans', sans-serif; font-size: 14px; line-height: 1.8;
  white-space: pre-wrap; color: var(--text-dark);
}

/* PRODUCTS */
.products-wrap { padding-top: 8px; }
.product-grid {
  display: grid; grid-template-columns: repeat(auto-fill, minmax(160px, 1fr));
  gap: 16px; margin-top: 24px;
}
.product-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: var(--radius); box-shadow: var(--shadow-sm);
  overflow: hidden; transition: transform 0.18s, box-shadow 0.18s;
}
.product-card:hover { transform: translateY(-3px); box-shadow: var(--shadow-md); }
.product-img-wrap {
  background: var(--bg); padding: 14px;
  display: flex; align-items: center; justify-content: center; height: 130px;
}
.product-img { width: 100%; height: 100%; object-fit: contain; }
.product-info { padding: 12px; }
.product-name { font-weight: 600; font-size: 13px; color: var(--text-dark); margin-bottom: 10px; }
.product-actions { display: flex; gap: 6px; }
.btn-buy {
  flex: 1; background: #ff8c00; color: white; text-decoration: none;
  text-align: center; padding: 6px 8px; border-radius: 7px;
  font-size: 12px; font-weight: 600; transition: background 0.16s;
}
.btn-buy:hover { background: #cc7000; }
.btn-view {
  flex: 1; background: #2c78c7; color: white; text-decoration: none;
  text-align: center; padding: 6px 8px; border-radius: 7px;
  font-size: 12px; font-weight: 600; transition: background 0.16s;
}
.btn-view:hover { background: #1a5a9e; }

/* HOW IT WORKS */
.how-section {
  background: var(--white); border-top: 1px solid var(--border);
  padding: 72px 24px; text-align: center;
}
.how-section .section-label, .how-section .section-title { text-align: center; }
.how-section .section-title { margin-bottom: 40px; }
.steps-grid {
  display: grid; grid-template-columns: repeat(3, 1fr);
  gap: 24px; max-width: 820px; margin: 0 auto;
}
.step-card {
  padding: 28px 20px; border: 1px solid var(--border);
  border-radius: var(--radius-lg); background: var(--bg);
  transition: box-shadow 0.18s;
}
.step-card:hover { box-shadow: var(--shadow-md); }
.step-num { font-family: 'Fraunces', serif; font-size: 13px; font-weight: 600; color: var(--text-light); margin-bottom: 8px; }
.step-icon { font-size: 32px; margin-bottom: 12px; }
.step-card h4 { font-size: 15px; font-weight: 600; margin-bottom: 8px; color: var(--text-dark); }
.step-card p { font-size: 13px; color: var(--text-mid); line-height: 1.6; }

/* FOOTER */
footer { background: var(--green-dark); color: rgba(255,255,255,0.75); padding: 40px 24px 28px; }
.footer-inner {
  max-width: 1100px; margin: auto;
  display: flex; flex-direction: column; align-items: center; gap: 16px; text-align: center;
}
.footer-brand { display: flex; flex-direction: column; align-items: center; gap: 6px; }
.footer-brand .logo-text { font-family: 'Fraunces', serif; font-size: 18px; font-weight: 600; color: white; }
.footer-tagline { font-size: 12px; color: rgba(255,255,255,0.5); }
.footer-links { display: flex; gap: 24px; }
.footer-links a {
  color: rgba(255,255,255,0.65); text-decoration: none;
  font-size: 13px; font-weight: 500; transition: color 0.16s;
}
.footer-links a:hover { color: white; }
.footer-copy { font-size: 12px; color: rgba(255,255,255,0.35); }

/* MODALS */
.modal-overlay {
  display: none; position: fixed; inset: 0; z-index: 999;
  background: rgba(10,30,18,0.55); backdrop-filter: blur(4px);
  align-items: center; justify-content: center; padding: 20px;
}
.modal-overlay.active { display: flex; }

.modal-box {
  background: var(--white); border-radius: var(--radius-lg);
  box-shadow: 0 20px 60px rgba(0,0,0,0.18);
  padding: 40px; max-width: 700px; width: 100%;
  max-height: 90vh; overflow-y: auto;
  position: relative; animation: modalIn 0.22s ease;
}
.modal-box--sm { max-width: 460px; }
@keyframes modalIn { from { opacity: 0; transform: translateY(16px); } to { opacity: 1; transform: none; } }

.modal-close {
  position: absolute; top: 16px; right: 18px;
  background: var(--bg); border: 1px solid var(--border);
  width: 32px; height: 32px; border-radius: 50%;
  font-size: 14px; cursor: pointer; color: var(--text-mid);
  display: flex; align-items: center; justify-content: center;
  transition: background 0.16s;
}
.modal-close:hover { background: var(--border); }

.modal-icon { font-size: 36px; margin-bottom: 12px; }
.modal-title {
  font-family: 'Fraunces', serif; font-size: 26px;
  color: var(--text-dark); margin-bottom: 10px;
}
.modal-lead { color: var(--text-mid); font-size: 15px; line-height: 1.6; }
.modal-divider { height: 1px; background: var(--border); margin: 24px 0; }
.modal-footer-note { font-size: 12px; color: var(--text-light); text-align: center; }

/* About grid */
.about-grid {
  display: grid; grid-template-columns: 1fr 1fr; gap: 20px;
}
.about-block {
  background: var(--bg); border: 1px solid var(--border);
  border-radius: var(--radius); padding: 18px;
}
.about-icon { font-size: 24px; display: block; margin-bottom: 8px; }
.about-block h4 { font-size: 14px; font-weight: 600; color: var(--text-dark); margin-bottom: 6px; }
.about-block p { font-size: 13px; color: var(--text-mid); line-height: 1.6; }

/* Contact rows */
.contact-row {
  display: flex; align-items: flex-start; gap: 14px;
  padding: 14px 0; border-bottom: 1px solid var(--border);
}
.contact-row:last-of-type { border-bottom: none; }
.contact-icon { font-size: 22px; margin-top: 2px; }
.contact-label { font-size: 11px; font-weight: 700; letter-spacing: 0.07em; text-transform: uppercase; color: var(--text-light); margin-bottom: 3px; }
.contact-value { font-size: 15px; font-weight: 600; color: var(--text-dark); }
.contact-email {
  font-size: 15px; font-weight: 600; color: var(--green-main);
  text-decoration: none; word-break: break-all;
}
.contact-email:hover { text-decoration: underline; }

/* RESPONSIVE */
@media (max-width: 768px) {
  .hero { grid-template-columns: 1fr; padding: 48px 20px 40px; text-align: center; }
  .hero-visual { display: none; }
  .hero-title { font-size: 32px; }
  .hero-sub { margin: 0 auto 28px; }
  .results-grid { grid-template-columns: 1fr; }
  .steps-grid { grid-template-columns: 1fr; }
  nav { display: none; }
  .how-section { padding: 48px 20px; }
  .about-grid { grid-template-columns: 1fr; }
}

/* ── LANGUAGE SELECTOR ───────────────────────────────────────────────── */
.lang-wrap { position: relative; }

.lang-btn {
  background: var(--white);
  border: 1.5px solid var(--border);
  padding: 7px 14px;
  border-radius: 8px;
  font-size: 13px;
  font-weight: 600;
  color: var(--green-dark);
  cursor: pointer;
  transition: border-color 0.18s, background 0.18s;
  white-space: nowrap;
}
.lang-btn:hover { border-color: var(--green-main); background: var(--green-light); }

.lang-menu {
  display: none;
  position: absolute;
  top: calc(100% + 6px);
  right: 0;
  background: var(--white);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  box-shadow: var(--shadow-md);
  min-width: 150px;
  z-index: 200;
  overflow: hidden;
}
.lang-menu.open { display: block; }

.lang-menu button {
  display: block;
  width: 100%;
  background: none;
  border: none;
  padding: 10px 16px;
  text-align: left;
  font-size: 13px;
  font-weight: 500;
  color: var(--text-dark);
  cursor: pointer;
  transition: background 0.14s;
}
.lang-menu button:hover { background: var(--green-light); color: var(--green-main); }

/* ── TRANSLATE BAR ───────────────────────────────────────────────────── */
.translate-bar {
  display: flex;
  align-items: center;
  gap: 12px;
  flex-wrap: wrap;
  margin-bottom: 20px;
  padding: 12px 16px;
  background: var(--white);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  font-size: 13px;
  font-weight: 500;
  color: var(--text-mid);
}
.translate-btns { display: flex; gap: 6px; flex-wrap: wrap; }
.tr-btn {
  background: var(--bg);
  border: 1.5px solid var(--border);
  padding: 5px 12px;
  border-radius: 20px;
  font-size: 12px;
  font-weight: 600;
  color: var(--text-mid);
  cursor: pointer;
  transition: all 0.16s;
}
.tr-btn:hover { border-color: var(--green-main); color: var(--green-main); }
.tr-btn.active { background: var(--green-main); border-color: var(--green-main); color: white; }

.translate-status {
  margin-left: auto;
  font-size: 12px;
  color: var(--text-light);
  font-style: italic;
}

/* ── ERROR BANNER ────────────────────────────────────────────────────── */
.error-banner {
  background: #fdecea;
  border: 1px solid #f5c6cb;
  border-left: 4px solid #c0392b;
  color: #c0392b;
  padding: 12px 18px;
  border-radius: var(--radius);
  font-size: 14px;
  font-weight: 500;
  margin-bottom: 20px;
}

/* ── ADVICE HEADER STATUS ────────────────────────────────────────────── */
.advice-header {
  display: flex;
  align-items: center;
  gap: 10px;
  padding: 18px 24px;
  border-bottom: 1px solid var(--border);
  background: var(--green-light);
  flex-wrap: wrap;
}
.advice-header .translate-status { margin-left: auto; font-size: 12px; color: var(--text-mid); }

@media (max-width: 768px) {
  .translate-bar { flex-direction: column; align-items: flex-start; }
  .lang-menu { right: auto; left: 0; }
}

/* ── NOT A CROP CARD ───────────────────────────────────────────────── */
.not-crop-card {
  background: var(--white);
  border: 2px solid #e74c3c;
  border-left: 6px solid #e74c3c;
  border-radius: var(--radius-lg);
  padding: 32px 28px;
  margin-bottom: 32px;
  box-shadow: 0 4px 20px rgba(231,76,60,0.12);
  text-align: center;
}
.not-crop-icon {
  font-size: 48px;
  margin-bottom: 12px;
}
.not-crop-title {
  font-family: 'Fraunces', serif;
  font-size: 22px;
  color: #c0392b;
  margin-bottom: 10px;
}
.not-crop-msg {
  color: var(--text-mid);
  font-size: 15px;
  margin-bottom: 20px;
  max-width: 520px;
  margin-left: auto;
  margin-right: auto;
}
.not-crop-preview {
  max-width: 260px;
  max-height: 200px;
  object-fit: cover;
  border-radius: var(--radius);
  border: 2px solid #f5c6cb;
  margin-bottom: 20px;
  display: block;
  margin-left: auto;
  margin-right: auto;
  opacity: 0.75;
}
.not-crop-tips {
  background: #fff5f5;
  border: 1px solid #f5c6cb;
  border-radius: var(--radius);
  padding: 16px 20px;
  text-align: left;
  max-width: 420px;
  margin: 0 auto 8px;
  font-size: 14px;
  color: var(--text-dark);
}
.not-crop-tips p { margin-bottom: 8px; }
.not-crop-tips ul {
  list-style: none;
  padding: 0;
  margin: 0;
}
.not-crop-tips ul li {
  padding: 4px 0;
  font-size: 13px;
  color: var(--text-mid);
}

/* ── NOT-A-CROP CARD ─────────────────────────────────────────────────── */
.not-crop-card {
  background: var(--white);
  border: 2px solid #e74c3c;
  border-left: 6px solid #e74c3c;
  border-radius: var(--radius-lg);
  padding: 28px;
  margin-bottom: 32px;
  box-shadow: 0 4px 20px rgba(231,76,60,0.10);
}

.not-crop-header {
  display: flex;
  align-items: flex-start;
  gap: 16px;
  margin-bottom: 24px;
}
.not-crop-icon {
  font-size: 40px;
  line-height: 1;
  flex-shrink: 0;
}
.not-crop-title {
  font-family: 'Fraunces', serif;
  font-size: 20px;
  color: #c0392b;
  margin-bottom: 6px;
}
.not-crop-subtitle {
  font-size: 14px;
  color: var(--text-mid);
  line-height: 1.5;
}

.not-crop-body {
  display: grid;
  grid-template-columns: 220px 1fr;
  gap: 24px;
  margin-bottom: 24px;
}
.not-crop-preview-wrap { text-align: center; }
.not-crop-label {
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 0.07em;
  text-transform: uppercase;
  color: var(--text-light);
  margin-bottom: 8px;
}
.not-crop-preview {
  width: 100%;
  max-height: 180px;
  object-fit: cover;
  border-radius: var(--radius);
  border: 1px solid var(--border);
  opacity: 0.7;
}

.not-crop-info { display: flex; flex-direction: column; gap: 4px; }

/* Stat bars */
.not-crop-stat {
  display: flex;
  align-items: center;
  gap: 10px;
}
.stat-name {
  font-size: 13px;
  font-weight: 600;
  color: var(--text-dark);
  min-width: 160px;
}
.stat-bar-wrap {
  flex: 1;
  background: var(--border);
  border-radius: 20px;
  height: 8px;
  overflow: hidden;
}
.stat-bar {
  display: block;
  height: 100%;
  border-radius: 20px;
  transition: width 0.5s ease;
}
.stat-val {
  font-size: 13px;
  font-weight: 700;
  min-width: 44px;
  text-align: right;
}
.stat-low { color: #e74c3c; }
.stat-mid { color: #f39c12; }
.stat-ok  { color: var(--green-main); }
.stat-note {
  font-size: 12px;
  color: var(--text-light);
  margin: 4px 0 8px 170px;
}

/* Tips box */
.not-crop-tips {
  background: #fff8f8;
  border: 1px solid #fad7d3;
  border-radius: var(--radius);
  padding: 14px 16px;
  margin-top: 12px;
}
.tips-title {
  font-size: 13px;
  font-weight: 700;
  color: var(--text-dark);
  margin-bottom: 8px;
}
.not-crop-tips ul {
  list-style: none;
  padding: 0;
  margin: 0;
  display: flex;
  flex-direction: column;
  gap: 5px;
}
.not-crop-tips li {
  font-size: 13px;
  color: var(--text-mid);
  line-height: 1.5;
}
.not-crop-btn {
  display: inline-block;
  background: #e74c3c;
}
.not-crop-btn:hover { background: #c0392b; }

@media (max-width: 768px) {
  .not-crop-body { grid-template-columns: 1fr; }
  .stat-name { min-width: 120px; }
  .stat-note { margin-left: 0; }
}


In [ ]:
!pkill -f flask || echo "No flask running"
!pkill -f ngrok || echo "No ngrok running"



In [ ]:
# 🔎 List processes using port 5000
!lsof -i :5000

In [ ]:
# ❌ Kill process by PID (replace 12345 with actual PID from previous cell)
!killall ngrok

In [ ]:
# ============================
# ✅ Restart Flask in background
# ============================
!pkill -f app.py || echo "No existing app running"
!nohup python app.py > flask.log 2>&1 &

In [ ]:

# ============================
# ✅ Start ngrok tunnel
# ============================
from pyngrok import ngrok, conf

conf.get_default().auth_token = "3B6rz2JHAuad59K4JEH4qFIdHRN_6w7VGXyKSdHKTmotQyrQh"   # 🔑 put your token here
public_url = ngrok.connect(5000)
print("🌍 Public URL:", public_url)


In [ ]:
!tail -n 50 flask.log

In [ ]:
from huggingface_hub import snapshot_download

# Explicitly download the model before running the app to avoid background timeouts
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
print(f"Downloading {model_id}...")

# This ensures the model is cached in /root/.cache/huggingface
snapshot_download(
    repo_id=model_id,
    allow_patterns=["*.json", "*.safetensors", "*.model"],
    ignore_patterns=["*.msgpack", "*.h5"]
)
print("\n✅ Model download complete!")

In [ ]:
!ls "/content/drive/My Drive/vit_model/plant_vit_base.pth"
!ls "/content/drive/My Drive/vit_model/class_names.json"